In [ ]:
import re
import pandas as pd

# 1) Path to your Sitemap log file
logfile = r"D:\sitemap_PSMA\sitemap_PSMA.log"

# 2) Read the entire log
with open(logfile, "r") as f:
    txt = f.read()

# 3) Regex to capture each Mapping block, its center/ranges, and the following SiteScore
pattern = re.compile(
    r"-- Mapping site\s+(\d+).*?"                                      # site number
    r"ligand xcm,ycm,zcm,xrange,yrange,zrange\s*"
    r"([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)"  # centers + half-ranges
    r".*?SiteScore.*?\n\s*([-\d\.]+)",                                   # capture the first number on the SiteScore line
    re.DOTALL
)

matches = pattern.findall(txt)

# 4) Build DataFrame
df = pd.DataFrame(matches, columns=[
    "site_rank",
    "center_x","center_y","center_z",
    "xrange","yrange","zrange",
    "score"
]).astype({
    "site_rank": int,
    "center_x": float, "center_y": float, "center_z": float,
    "xrange": float,   "yrange": float,   "zrange": float,
    "score": float
})

# 5) Keep only the top-2 sites by rank
top2 = df[df["site_rank"].isin([1, 2])].sort_values("site_rank")

# 6) Compute full box sizes for Vina
top2["size_x"] = top2["xrange"] * 2
top2["size_y"] = top2["yrange"] * 2
top2["size_z"] = top2["zrange"] * 2

# 7) Annotate with filename and a simple site label
top2["filename"] = logfile
top2["site"]     = top2["site_rank"].apply(lambda x: f"site_{x}")

# 8) Reorder columns
out = top2[[
    "filename","site","site_rank","score",
    "center_x","center_y","center_z",
    "size_x","size_y","size_z"
]]

# 9) Write out
out_csv = r"D:\sitemap_PSMA\top2_sites.csv"
out.to_csv(out_csv, index=False)

print(f"Wrote top 2 sites (with scores) to {out_csv}\n")
print(out)


In [ ]:
import re
from pathlib import Path
import pandas as pd

# -----------------------------
# 1) Configure your directories
# -----------------------------
# Change this to the directory that contains all your *_prep folders
base_dir = Path(r"D:\sitemap_all_proteins")

# Where to write the final master CSV
out_csv = base_dir / "all_proteins_top2_sites.csv"

# -----------------------------------
# 2) Precompile the sitemap regex
# -----------------------------------
pattern = re.compile(
    r"-- Mapping site\s+(\d+).*?"                                      # site number
    r"ligand xcm,ycm,zcm,xrange,yrange,zrange\s*"
    r"([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)"  # center & half-ranges
    r".*?SiteScore.*?\n\s*([-\d\.]+)",                                   # the first number on the SiteScore line
    re.DOTALL
)

# --------------------------------------------------
# 3) Loop over each folder, parse, and collect rows
# --------------------------------------------------
rows = []
for prep_folder in base_dir.iterdir():
    if not prep_folder.is_dir() or not prep_folder.name.endswith("_prep"):
        continue
    
    protein = prep_folder.name.rsplit("_prep", 1)[0]
    # find the first .log file in that folder
    log_files = list(prep_folder.glob("*.log"))
    if not log_files:
        print(f"⚠️  No .log in {prep_folder.name}, skipping")
        continue
    logfile = log_files[0]
    txt = logfile.read_text()

    # apply regex
    matches = pattern.findall(txt)
    if not matches:
        print(f"⚠️  No site matches in {logfile}, skipping")
        continue

    # build a DataFrame for this one protein
    df = pd.DataFrame(matches, columns=[
        "site_rank",
        "center_x","center_y","center_z",
        "xrange","yrange","zrange",
        "score"
    ]).astype({
        "site_rank": int,
        "center_x": float, "center_y": float, "center_z": float,
        "xrange": float,   "yrange": float,   "zrange": float,
        "score": float
    })

    # keep only the top 2 by rank
    top2 = df[df.site_rank.isin([1, 2])].sort_values("site_rank").copy()
    if top2.empty:
        continue

    # compute full box sizes
    top2["size_x"] = top2["xrange"] * 2
    top2["size_y"] = top2["yrange"] * 2
    top2["size_z"] = top2["zrange"] * 2

    # annotate
    top2["protein"]  = protein
    top2["logfile"]  = str(logfile)
    top2["site_id"]  = top2.site_rank.apply(lambda n: f"site_{n}")

    # select / reorder
    for _, row in top2.iterrows():
        rows.append({
            "protein":    row["protein"],
            "site_id":    row["site_id"],
            "site_rank":  row["site_rank"],
            "score":      row["score"],
            "center_x":   row["center_x"],
            "center_y":   row["center_y"],
            "center_z":   row["center_z"],
            "size_x":     row["size_x"],
            "size_y":     row["size_y"],
            "size_z":     row["size_z"],
            "logfile":    row["logfile"],
        })

# ---------------------------------------
# 4) Concatenate everything and save CSV
# ---------------------------------------
if rows:
    master_df = pd.DataFrame(rows)
    master_df.to_csv(out_csv, index=False)
    print(f"✅  Wrote {len(master_df)} rows to {out_csv}")
    display(master_df.head(10))
else:
    print("⚠️  No data extracted—check your folders/logfiles.")
